[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C07_ML_Foundations_Course/02_probability_statistics_info/02_probability_statistics_info.ipynb)

# 02 · 概率统计与信息论

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在真实数据上把 MLE、假设检验、熵/KL/交叉熵全部算一遍——并亲手算出一个语言模型的困惑度。

**你将完成：**
1. 高斯 MLE：在真实企鹅体重上手算 `μ̂, σ̂²`
2. 置换检验：Adelie vs Gentoo 体重差异显著吗
3. 熵 / 交叉熵 / 困惑度：在 **tiny-shakespeare** 真实文本上算 bigram 语言模型的 loss
4. KL 散度：验证 `KL≥0` 与不对称性

> 数据：Palmer Penguins（体重）+ Karpathy 的 tiny-shakespeare（莎士比亚全集 ~1MB 文本）。

## 0 · 加载真实数据

In [ ]:
import os, urllib.request
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
CACHE = os.path.expanduser("~/.ml_foundations_data"); os.makedirs(CACHE, exist_ok=True)
def fetch(url, fname):
    p = os.path.join(CACHE, fname)
    if not os.path.exists(p): urllib.request.urlretrieve(url, p)
    return p

peng = pd.read_csv(fetch("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv","penguins.csv"))
peng = peng.dropna(subset=["body_mass_g","species"])
adelie = peng[peng.species=="Adelie"].body_mass_g.to_numpy(float)
gentoo = peng[peng.species=="Gentoo"].body_mass_g.to_numpy(float)
print(f"Adelie n={len(adelie)} 均值={adelie.mean():.0f}g | Gentoo n={len(gentoo)} 均值={gentoo.mean():.0f}g")

txt = open(fetch("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shakespeare.txt")).read()
print(f"\nshakespeare: {len(txt):,} 字符, {len(set(txt))} 种唯一字符")
print(repr(txt[:80]))

## 1 · 高斯 MLE（在真实体重上）

闭式解：$\hat\mu=\bar x$，$\hat\sigma^2=\frac1n\sum(x-\bar x)^2$。我们手算，并验证它确实让对数似然最大。

In [ ]:
x = adelie
mu_hat = x.mean(); var_hat = ((x-mu_hat)**2).mean()
print(f"MLE: μ̂={mu_hat:.1f}g  σ̂={np.sqrt(var_hat):.1f}g")

def loglik(mu, var, x):
    return np.sum(-0.5*np.log(2*np.pi*var) - (x-mu)**2/(2*var))

# 验证：在 μ̂ 附近扰动，loglik 都更小
base = loglik(mu_hat, var_hat, x)
worse = [loglik(mu_hat+d, var_hat, x) for d in (-50, 50)]
print(f"loglik(μ̂)={base:.1f}  >  两侧扰动 {worse[0]:.1f}, {worse[1]:.1f}  => μ̂ 确为极大 ✓")

## 2 · 置换检验：体重差异显著吗

Adelie 与 Gentoo 体重差 ~1300g。但这是真差异还是抽样噪声？置换检验：把两组标签洗牌很多次，
看真实均值差在零分布里有多极端。

In [ ]:
obs = gentoo.mean() - adelie.mean()
pool = np.concatenate([adelie, gentoo]); n_a = len(adelie)
rng = np.random.default_rng(0); N = 10000
perm = np.empty(N)
for i in range(N):
    s = rng.permutation(pool)
    perm[i] = s[n_a:].mean() - s[:n_a].mean()
p = (np.abs(perm) >= np.abs(obs)).mean()
print(f"观测差 Δ={obs:.0f}g   置换零分布 std={perm.std():.0f}g")
print(f"p 值 = {p:.5f}  => {'极显著（远超噪声）' if p<0.001 else '不显著'}")

## 3 · 熵、交叉熵、困惑度（真实文本上的 bigram 语言模型）

构造一个 bigram 模型 $q(c_{t}\mid c_{t-1})$（用前半文本统计），在后半文本上算交叉熵与困惑度。
**这就是最小的语言模型评测**：困惑度越低，模型对真实文本越"不惊讶"。

In [ ]:
chars = sorted(set(txt)); V = len(chars); stoi = {c:i for i,c in enumerate(chars)}
ids = np.array([stoi[c] for c in txt])
half = len(ids)//2
train, test = ids[:half], ids[half:]

# bigram 计数 + 拉普拉斯平滑（避免 0 概率 -> inf loss）
counts = np.ones((V, V))  # +1 smoothing
for a, b in zip(train[:-1], train[1:]): counts[a, b] += 1
Q = counts / counts.sum(1, keepdims=True)     # 行归一化 = q(next|prev)

# 测试集交叉熵（自然对数 -> nats）
logq = np.log(Q[test[:-1], test[1:]])
ce = -logq.mean()
ppl = np.exp(ce)
# 对比：均匀分布的交叉熵 = log(V)
print(f"字符表大小 V={V}")
print(f"bigram 交叉熵 = {ce:.4f} nats/char   困惑度 PPL = {ppl:.2f}")
print(f"均匀基线 PPL = {V:.0f}  => bigram 把'每步犹豫'从 {V} 个降到 {ppl:.1f} 个")

# 单字符熵（unigram 经验熵）
p_uni = np.bincount(train, minlength=V)/len(train)
H = -(p_uni[p_uni>0]*np.log(p_uni[p_uni>0])).sum()
print(f"unigram 经验熵 H = {H:.4f} nats  (PPL={np.exp(H):.1f})  => bigram 比 unigram 更好")

## 4 · MAP = 加先验的 MLE（高斯先验 = L2 正则）· 以及 KL 的非对称性

在 MLE 上加一个先验 $p(\theta)$、最大化后验，就是 **MAP**。关键等价：**零均值高斯先验的 MAP = L2 正则化**，拉普拉斯先验 = L1。下面在真实企鹅体重上把这个「往先验收缩（shrinkage）」算出来——先验越强、数据越少，估计就越被先验拉走；先验趋于无信息时退回 MLE。最后再用两个真实分布验证 **KL 的非对称性**（forward vs reverse KL，正是知识蒸馏里二者行为不同的根源）。

In [ ]:
# MAP：给均值加一个高斯先验 N(m0, τ²)，看后验均值如何在「数据」与「先验」间折中
# 已知方差 σ² 时，高斯似然 + 高斯先验的后验均值有闭式解（共轭）：
#   μ_MAP = (n/σ² · x̄ + 1/τ² · m0) / (n/σ² + 1/τ²)
# 这正是「带 L2 正则把估计往先验拉」的概率版本——τ→∞（无先验）时退回 MLE
x = adelie
n = len(x); xbar = x.mean(); sigma2 = ((x - xbar)**2).mean()  # 用 MLE 方差当已知 σ²
m0 = 3000.0        # 先验认为体重均值约 3000g（故意偏离数据真值 ~3700g）

def map_mean(tau2):
    prec_data = n / sigma2          # 数据精度（越多数据越自信）
    prec_prior = 1.0 / tau2         # 先验精度（先验越紧越自信）
    return (prec_data * xbar + prec_prior * m0) / (prec_data + prec_prior)

mu_weak  = map_mean(tau2=1e8)   # 极弱先验 -> 几乎等于 MLE
mu_strong = map_mean(tau2=2e3)  # 较强先验 -> 被往 m0=3000 拉
print(f"MLE 均值 x̄          = {xbar:.1f}g")
print(f"MAP（弱先验 τ²=1e8） = {mu_weak:.1f}g   => 几乎等于 MLE（先验几乎不起作用）")
print(f"MAP（强先验 τ²=2e3） = {mu_strong:.1f}g   => 被先验 m0={m0:.0f} 往下拉")

assert abs(mu_weak - xbar) < 1.0, "弱先验下 MAP 应收敛到 MLE"
assert m0 < mu_strong < xbar, "强先验下 MAP 应落在先验与数据之间（shrinkage）"
print("MAP=正则化 验证通过 ✓  —— L2 正则 = 零均值高斯先验的 MAP，这里把它在真实数据上算了出来")

In [ ]:
# forward KL vs reverse KL（在两个真实企鹅 flipper 分布上）
# 用真实数据构造两个归一化直方图，展示 KL 的非对称性——这正是蒸馏里 forward/reverse KL 行为不同的根源
fa = peng[peng.species=="Adelie"].flipper_length_mm.dropna().to_numpy()
fg = peng[peng.species=="Gentoo"].flipper_length_mm.dropna().to_numpy()
bins = np.linspace(170, 235, 14)
pa = np.histogram(fa, bins)[0] + 1.0; pa /= pa.sum()   # +1 平滑，保证处处 >0（否则 KL 可能 inf）
pg = np.histogram(fg, bins)[0] + 1.0; pg /= pg.sum()

def kl_demo(p, q):
    return float(np.sum(p*np.log(p/q)))

fwd = kl_demo(pa, pg)   # forward: D(A||G)，对 A 有质量而 G 没有的地方惩罚重 -> mode-covering
rev = kl_demo(pg, pa)   # reverse: D(G||A)
print(f"D_KL(A‖G) = {fwd:.4f} nats   D_KL(G‖A) = {rev:.4f} nats")
print(f"非对称差 |fwd - rev| = {abs(fwd-rev):.4f}  => KL 不是距离（不对称、不满足三角不等式）")
print(f"自身 KL: D(A‖A) = {kl_demo(pa,pa):.2e} ≈ 0  ✓（Gibbs 不等式取等仅当 p=q）")

assert fwd > 0 and rev > 0, "p≠q 时 KL 严格 >0（Gibbs 不等式）"
assert abs(fwd - rev) > 1e-6, "forward 与 reverse KL 应不相等（非对称）"
assert kl_demo(pa, pa) < 1e-12, "D(p‖p)=0"
print("KL 性质验证通过 ✓  —— 训练 LLM 最小化 forward KL（= 交叉熵，因 H(p) 与参数无关）")

---
## ✏️ 练习区

### ✏️ 练习 1：高斯 MLE

实现 `gaussian_mle(x)` 返回 `(mu_hat, var_hat)`（用 MLE 的 `/n` 方差，不是 `/(n-1)`）。

In [ ]:
def gaussian_mle(x):
    # TODO: 返回 (均值, MLE方差)
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
m, v = gaussian_mle(gentoo)
assert abs(m - gentoo.mean()) < 1e-9
assert abs(v - gentoo.var()) < 1e-9   # np.var 默认就是 /n
assert abs(v - gentoo.var(ddof=1)) > 1, "必须是 /n 的有偏方差，不是 /(n-1)"
print("练习 1 通过 ✓")


### ✏️ 练习 2：置换检验 p 值

实现 `perm_test(a, b, n_perm, seed)`，返回双侧 p 值（统计量 = 两组均值差）。

In [ ]:
def perm_test(a, b, n_perm=10000, seed=0):
    # TODO: obs=mean(b)-mean(a)；洗牌 pool n_perm 次算零分布；p=比例(|perm|>=|obs|)
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
p_real = perm_test(adelie, gentoo, 5000, seed=1)
assert p_real < 0.01, "真实体重差异应显著"
# 同一组对自己：应不显著
half = len(adelie)//2
p_null = perm_test(adelie[:half], adelie[half:], 5000, seed=1)
assert p_null > 0.05, "同分布两半不应显著"
print(f"练习 2 通过 ✓  显著对 p={p_real:.4f}  无差异对 p={p_null:.3f}")


### ✏️ 练习 3：交叉熵与困惑度

实现 `cross_entropy_ppl(Q, prev, nxt)`：给转移矩阵 `Q` 和测试序列，返回 `(交叉熵_nats, 困惑度)`。

In [ ]:
def cross_entropy_ppl(Q, prev, nxt):
    # TODO: ce = -mean(log Q[prev, nxt])；ppl = exp(ce)
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
ce2, ppl2 = cross_entropy_ppl(Q, test[:-1], test[1:])
assert abs(ce2 - ce) < 1e-9 and abs(ppl2 - np.exp(ce)) < 1e-6
# 完美模型（用测试集自己的转移、对角占优）困惑度应更低；均匀矩阵 PPL≈V
Qu = np.full((V, V), 1/V)
_, ppl_u = cross_entropy_ppl(Qu, test[:-1], test[1:])
assert abs(ppl_u - V) < 1e-6
assert ppl2 < ppl_u, "bigram 应优于均匀"
print(f"练习 3 通过 ✓  bigram PPL={ppl2:.1f} < 均匀 PPL={ppl_u:.0f}")


### ✏️ 练习 4：KL 散度与它的性质

实现 `kl(p, q)` $=\sum p\log(p/q)$。然后验证 `KL≥0` 与**不对称**（用两个真实分布：
Adelie 与 Gentoo 的 flipper 长度直方图）。

In [ ]:
def kl(p, q):
    # TODO: 只对 p>0 求和；用 np.log；假设 q>0
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
# 用真实 flipper 长度做两个归一化直方图（同样的 bins）
fa = peng[peng.species=="Adelie"].flipper_length_mm.dropna().to_numpy()
fg = peng[peng.species=="Gentoo"].flipper_length_mm.dropna().to_numpy()
bins = np.linspace(170, 235, 14)
pa = np.histogram(fa, bins)[0] + 1.0; pa /= pa.sum()
pg = np.histogram(fg, bins)[0] + 1.0; pg /= pg.sum()
assert kl(pa, pa) < 1e-12, "KL(p||p)=0"
assert kl(pa, pg) > 0, "KL≥0 且 p≠q 时 >0"
assert abs(kl(pa, pg) - kl(pg, pa)) > 1e-6, "KL 不对称"
print(f"练习 4 通过 ✓  KL(A||G)={kl(pa,pg):.3f} ≠ KL(G||A)={kl(pg,pa):.3f}")


---
## 📖 参考答案

In [ ]:
# 练习 1
def gaussian_mle(x):
    x = np.asarray(x, float); mu = x.mean(); return mu, ((x-mu)**2).mean()
print("练习 1 ✓")

In [ ]:
# 练习 2
def perm_test(a, b, n_perm=10000, seed=0):
    rng = np.random.default_rng(seed)
    obs = b.mean() - a.mean(); pool = np.concatenate([a, b]); na = len(a)
    cnt = 0
    for _ in range(n_perm):
        s = rng.permutation(pool)
        if abs(s[na:].mean() - s[:na].mean()) >= abs(obs): cnt += 1
    return cnt / n_perm
print("练习 2 ✓")

In [ ]:
# 练习 3
def cross_entropy_ppl(Q, prev, nxt):
    ce = -np.log(Q[prev, nxt]).mean(); return ce, np.exp(ce)
print("练习 3 ✓")

In [ ]:
# 练习 4
def kl(p, q):
    p = np.asarray(p, float); q = np.asarray(q, float); m = p > 0
    return float(np.sum(p[m]*np.log(p[m]/q[m])))
print("练习 4 ✓ —— KL 不对称正是为什么 forward/reverse KL 在蒸馏里表现不同")